# Clasificación de `muestra_100.csv` con tu modelo fine-tuned de OpenAI

Esta notebook:

- lee un archivo local llamado `muestra_100.csv`
- llama a tu modelo `ft:gpt-3.5-turbo-0125:personal:hatemultiv4:9uouAqOs`
- agrega nuevas columnas con la salida del modelo GPT
- guarda un archivo nuevo con los resultados
- resume los tokens usados para estimar el costo después

Antes de correrla:

1. instala `openai`, `pandas` y `tqdm`
2. define tu API key en la variable de entorno `OPENAI_API_KEY`
3. verifica cuál es la columna de texto en tu CSV

La notebook usa `chat.completions.create(...)`, que sigue siendo una forma válida de invocar modelos `gpt-3.5-turbo` fine-tuned en la API.

In [ ]:
# Si hace falta, descomenta esta línea:
# !pip install -q openai pandas tqdm


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os
import json
import time
from typing import Any, Dict, List, Optional

import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

In [9]:
from dotenv import load_dotenv
load_dotenv()

# =========================
# Configuración principal
# =========================

MODEL_NAME = 'ft:gpt-3.5-turbo-0125:personal:hatemultiv4:9uouAqOs'
INPUT_CSV = 'DNU_Data/muestra_100.csv'
OUTPUT_CSV = 'muestra_100_con_gpt.csv'

# Si ya sabes el nombre exacto de la columna de texto, escríbelo acá.
# Si lo dejas en None, la notebook intentará detectarlo automáticamente.
TEXT_COLUMN = None

# Columna opcional con la etiqueta previa de otro clasificador.
# Solo se conserva; no hace falta usarla para inferencia.
EXISTING_LABEL_COLUMN = None

# Cantidad máxima de filas a procesar. Para tu archivo actual, 100 está bien.
MAX_ROWS = 100

# Para clasificación conviene mantener salida corta y estable.
TEMPERATURE = 0
MAX_TOKENS = 120
SLEEP_SECONDS = 0.15

client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

if not os.environ.get('OPENAI_API_KEY'):
    raise ValueError('No encuentro OPENAI_API_KEY en las variables de entorno.')

In [10]:
df = pd.read_csv(INPUT_CSV)
print(f'Filas: {len(df)}')
print('Columnas:')
print(list(df.columns))
df.head(3)

Filas: 100
Columnas:
['created_at', 'id', 'id_str', 'text', 'source', 'truncated', 'in_reply_to_status_id', 'in_reply_to_status_id_str', 'in_reply_to_user_id', 'quoted_status.place.place_type', 'quoted_status.place.name', 'retweeted_status.quoted_status.quoted_status_id', 'is_retweet', 'caracteres', 'text_pp', 'tokens', 'fecha', 'fecha_dia', 'CALLS', 'WOMEN', 'LGBTI', 'RACISM', 'CLASS', 'POLITICS', 'DISABLED', 'APPEARANCE', 'CRIMINAL', 'labels', 'n_labels', 'HATEFUL']


,created_at,id,id_str,text,source,truncated,in_reply_to_status_id,in_reply_to_status_id_str,in_reply_to_user_id,quoted_status.place.place_type,...,LGBTI,RACISM,CLASS,POLITICS,DISABLED,APPEARANCE,CRIMINAL,labels,n_labels,HATEFUL
0,Sat Mar 06 23:16:38 +0000 2021,1368339770686996480,1368339770686996480,Extranjeros participan de los destrozos realiz...,"<a href=""http://twitter.com/download/android"" ...",False,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,[],0,0
1,Fri Mar 05 17:33:11 +0000 2021,1367890949792272391,1367890949792272391,@pepegoyomr @CARLOSLGUERRERO Al contrario!!! E...,"<a href=""http://twitter.com/download/iphone"" r...",True,1.367887e+18,1.367887e+18,142093384.0,NaN,...,0,0,0,0,0,0,0,[],0,0
2,Fri Mar 05 22:02:06 +0000 2021,1367958627739258880,1367958627739258880,La diferencia con los kirchneristas ya no es s...,"<a href=""http://twitter.com/download/iphone"" r...",True,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,[],0,0


In [11]:
df = df[['id', 'text']].copy()
text_col = 'text'
print(f'Columnas: {list(df.columns)}')
print(f'Total filas: {len(df):,}')
df.head(3)

Columnas: ['id', 'text']
Total filas: 100


,id,text
0,1368339770686996480,Extranjeros participan de los destrozos realiz...
1,1367890949792272391,@pepegoyomr @CARLOSLGUERRERO Al contrario!!! E...
2,1367958627739258880,La diferencia con los kirchneristas ya no es s...


## Plantilla del prompt

Esta parte es la más probable de necesitar ajuste fino.

Como no veo aquí el formato exacto con el que entrenaste el modelo, la notebook le pide una salida JSON breve. Si al probar 5–10 casos ves que el modelo responde en otro formato, solo hay que editar esta celda.

In [12]:
LABELS = ["CALLS", "WOMEN", "LGBTI", "RACISM", "CLASS", "POLITICS", "DISABLED", "APPEARANCE", "CRIMINAL"]

SYSTEM_MESSAGE = (
    "Eres un clasificador multietiqueta de discurso de odio entrenado sobre textos en espanol. "
    "Dado un tweet, devuelve exclusivamente un JSON valido en una sola linea indicando "
    "que categorias de odio estan presentes. "
    "Categorias posibles: CALLS, WOMEN, LGBTI, RACISM, CLASS, POLITICS, DISABLED, APPEARANCE, CRIMINAL. "
    "Si no hay odio devuelve lista vacia. No agregues texto fuera del JSON."
)

# IMPORTANTE: las llaves literales del ejemplo JSON se escapan con {{ y }}
# para que .format(text=text) no las confunda con placeholders.
USER_TEMPLATE = """Clasifica el siguiente tweet y responde SOLO con JSON valido.
Estructura exacta: {{"labels": ["ETIQUETA1", "ETIQUETA2"]}}
Etiquetas validas: CALLS, WOMEN, LGBTI, RACISM, CLASS, POLITICS, DISABLED, APPEARANCE, CRIMINAL.
Sin odio: {{"labels": []}}

Tweet:
{text}"""

print("Etiquetas:", LABELS)
print()
print("System message:")
print(SYSTEM_MESSAGE)
print()
print("User template (preview con texto de prueba):")
print(USER_TEMPLATE.format(text="[tweet de ejemplo]"))


Etiquetas: ['CALLS', 'WOMEN', 'LGBTI', 'RACISM', 'CLASS', 'POLITICS', 'DISABLED', 'APPEARANCE', 'CRIMINAL']

System message:
Eres un clasificador multietiqueta de discurso de odio entrenado sobre textos en espanol. Dado un tweet, devuelve exclusivamente un JSON valido en una sola linea indicando que categorias de odio estan presentes. Categorias posibles: CALLS, WOMEN, LGBTI, RACISM, CLASS, POLITICS, DISABLED, APPEARANCE, CRIMINAL. Si no hay odio devuelve lista vacia. No agregues texto fuera del JSON.

User template (preview con texto de prueba):
Clasifica el siguiente tweet y responde SOLO con JSON valido.
Estructura exacta: {"labels": ["ETIQUETA1", "ETIQUETA2"]}
Etiquetas validas: CALLS, WOMEN, LGBTI, RACISM, CLASS, POLITICS, DISABLED, APPEARANCE, CRIMINAL.
Sin odio: {"labels": []}

Tweet:
[tweet de ejemplo]


In [13]:
def safe_json_loads(text: str) -> Optional[Dict[str, Any]]:
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find('{')
    end = text.rfind('}')
    if start != -1 and end != -1 and end > start:
        candidate = text[start:end+1]
        try:
            return json.loads(candidate)
        except Exception:
            return None
    return None


def classify_text(text: str) -> Dict[str, Any]:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        messages=[
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": USER_TEMPLATE.format(text=text)}
        ]
    )

    raw = response.choices[0].message.content
    parsed = safe_json_loads(raw) if raw else None
    usage = getattr(response, 'usage', None)

    return {
        'raw_response': raw,
        'parsed_json': parsed,
        'prompt_tokens': getattr(usage, 'prompt_tokens', None),
        'completion_tokens': getattr(usage, 'completion_tokens', None),
        'total_tokens': getattr(usage, 'total_tokens', None),
        'finish_reason': response.choices[0].finish_reason,
        'model_used': response.model,
    }

## Prueba corta antes de procesar todo

In [14]:
sample_text = str(df[text_col].iloc[0])
test_result = classify_text(sample_text)
test_result

{'raw_response': '{"labels": ["RACISM"]}',
 'parsed_json': {'labels': ['RACISM']},
 'prompt_tokens': 231,
 'completion_tokens': 8,
 'total_tokens': 239,
 'finish_reason': 'stop',
 'model_used': 'ft:gpt-3.5-turbo-0125:personal:hatemultiv4:9uouAqOs'}

Si la salida de arriba te gusta, corre la siguiente celda para procesar el archivo completo.

In [ ]:
work_df = df.copy().head(MAX_ROWS).reset_index(drop=True)

gpt_raw_responses: List[Optional[str]] = []
gpt_labels_json: List[Optional[str]] = []
gpt_notes: List[Optional[str]] = []
gpt_prompt_tokens: List[Optional[int]] = []
gpt_completion_tokens: List[Optional[int]] = []
gpt_total_tokens: List[Optional[int]] = []
gpt_finish_reason: List[Optional[str]] = []
gpt_error: List[Optional[str]] = []

for _, row in tqdm(work_df.iterrows(), total=len(work_df)):
    text_value = str(row[text_col]) if pd.notna(row[text_col]) else ''

    try:
        result = classify_text(text_value)
        parsed = result['parsed_json'] or {}

        labels = parsed.get('labels') if isinstance(parsed, dict) else None
        notes = None if isinstance(parsed, dict) else None

        gpt_raw_responses.append(result['raw_response'])
        gpt_labels_json.append(json.dumps(labels, ensure_ascii=False) if labels is not None else None)
        gpt_notes.append(notes)
        gpt_prompt_tokens.append(result['prompt_tokens'])
        gpt_completion_tokens.append(result['completion_tokens'])
        gpt_total_tokens.append(result['total_tokens'])
        gpt_finish_reason.append(result['finish_reason'])
        gpt_error.append(None)

    except Exception as e:
        gpt_raw_responses.append(None)
        gpt_labels_json.append(None)
        gpt_notes.append(None)
        gpt_prompt_tokens.append(None)
        gpt_completion_tokens.append(None)
        gpt_total_tokens.append(None)
        gpt_finish_reason.append(None)
        gpt_error.append(str(e))

    time.sleep(SLEEP_SECONDS)

work_df['gpt_raw_response'] = gpt_raw_responses
work_df['gpt_labels_json'] = gpt_labels_json
work_df['gpt_notes'] = gpt_notes
work_df['gpt_prompt_tokens'] = gpt_prompt_tokens
work_df['gpt_completion_tokens'] = gpt_completion_tokens
work_df['gpt_total_tokens'] = gpt_total_tokens
work_df['gpt_finish_reason'] = gpt_finish_reason
work_df['gpt_error'] = gpt_error

work_df.to_csv(OUTPUT_CSV, index=False)
print(f'Archivo guardado en: {OUTPUT_CSV}')

In [ ]:

# ── 1. Recuperar text + HATEFUL (gold) desde el CSV original ─────────────────
df_gold = pd.read_csv(INPUT_CSV, usecols=['id', 'text', 'HATEFUL'])
df_gold = df_gold.rename(columns={'text': 'text_original'})

# ── 2. Unir con work_df (renombrando text para distinguirlos) ────────────────
results_df = work_df.rename(columns={'text': 'text_work'}).merge(df_gold, on='id', how='left')

# ── 3. Verificación del merge: comparar textos ───────────────────────────────
print("Verificación del merge (primeras 5 filas):")
display(results_df[['id', 'text_work', 'text_original']].head())

# ── 4. Predicción binaria GPT: 1 si devolvió al menos una etiqueta ───────────
def to_hateful(labels_json):
    if pd.isna(labels_json):
        return None  # llamada con error
    try:
        return int(len(json.loads(labels_json)) > 0)
    except Exception:
        return None

results_df['gpt_hateful'] = results_df['gpt_labels_json'].apply(to_hateful)

# ── 5. Benchmark ─────────────────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix

valid = results_df.dropna(subset=['gpt_hateful', 'HATEFUL'])
y_true = valid['HATEFUL'].astype(int)
y_pred = valid['gpt_hateful'].astype(int)

print()
print("=" * 50)
print("BENCHMARK: GPT fine-tuned vs. HATEFUL (gold)")
print("=" * 50)
print(f"Filas con predicción válida: {len(valid)} / {len(results_df)}")
print()
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))
print()
print(classification_report(y_true, y_pred, target_names=['NO ODIO', 'ODIO']))

# ── 6. Resumen de tokens ─────────────────────────────────────────────────────
summary = {
    'rows_processed': int(len(work_df)),
    'rows_with_error': int(work_df['gpt_error'].notna().sum()),
    'prompt_tokens_sum': int(pd.Series(gpt_prompt_tokens).fillna(0).sum()),
    'completion_tokens_sum': int(pd.Series(gpt_completion_tokens).fillna(0).sum()),
    'total_tokens_sum': int(pd.Series(gpt_total_tokens).fillna(0).sum()),
}
print("Tokens:", summary)


## Estimación de costo

Completa esta celda cuando tengas el precio vigente de inferencia para tu modelo o para la familia correspondiente en tu panel/API pricing.

La fórmula general es:

- costo input = `prompt_tokens_sum / 1_000_000 * precio_input`
- costo output = `completion_tokens_sum / 1_000_000 * precio_output`
- costo total = suma de ambos

Si más adelante quieres correr un dataset mucho más grande, esta misma notebook te sirve para obtener una muestra real de tokens por fila y extrapolar.

In [ ]:
# Ejemplo: reemplaza estos valores por los precios vigentes que quieras usar.
PRICE_INPUT_PER_1M = None
PRICE_OUTPUT_PER_1M = None

if PRICE_INPUT_PER_1M is not None and PRICE_OUTPUT_PER_1M is not None:
    input_cost = summary['prompt_tokens_sum'] / 1_000_000 * PRICE_INPUT_PER_1M
    output_cost = summary['completion_tokens_sum'] / 1_000_000 * PRICE_OUTPUT_PER_1M
    total_cost = input_cost + output_cost

    print('Costo estimado input:', round(input_cost, 6))
    print('Costo estimado output:', round(output_cost, 6))
    print('Costo estimado total:', round(total_cost, 6))
else:
    print('Todavía no cargaste los precios por 1M de tokens.')

## Sugerencias rápidas

- primero prueba 5 a 10 filas
- revisa si `gpt_labels_json` sale con el formato que esperas
- si el modelo fue entrenado con otra convención de etiquetas o con otra plantilla, adapta `USER_TEMPLATE`
- si luego quieres escalar a miles de filas, puede convenir Batch API